# 🧪 실습 1 — mAb 안정성 인과 네트워크 만들기 (PubMed → GPT → 그래프)

이 노트북 하나로 **논문 수집 → 관련 논문 골라내기 → 인과관계 추출 → 그래프 그리기**까지 직접 해봅니다.

```
STEP 1  PubMed에서 논문 초록 수집      (무료, 인터넷만 있으면 됨)
STEP 2  GPT로 '안정성과 관련된' 논문만 필터링   (OpenAI 키 필요 · 없으면 건너뜀)
STEP 3  GPT로 (원인 → 관계 → 결과) 추출        (OpenAI 키 필요 · 없으면 샘플 사용)
STEP 4  내가 만든 표를 네트워크 그래프로 시각화   (무료)
```

> 💡 **OpenAI 키가 없어도 괜찮아요.** STEP 2·3은 건너뛰고, 연구팀이 공개한 실제 데이터(샘플)로 STEP 4 그래프를 그릴 수 있습니다.
> 키가 있으면 아주 적은 양(논문 20편 정도, 수십 원 수준)만 써서 '내 그래프'를 직접 만들 수 있습니다.

> ⚠️ GATED **모델 학습(STEP 5)** 은 이 실습 범위가 아닙니다. 개념은 마지막 셀과 교안에서 설명합니다.

## 0. 준비 — 패키지 설치

`koreanize-matplotlib` 는 그래프의 **한글 깨짐(□□□)** 을 막아주는 라이브러리예요(Colab에 한글 폰트가 없어 필요).

In [ ]:
!pip install -q requests xmltodict pandas networkx matplotlib koreanize-matplotlib openai
print('설치 완료')

## STEP 1 — PubMed에서 논문 초록 수집

PubMed(미국 국립의학도서관)의 무료 API로 'mAb 안정성' 관련 논문 제목·초록을 가져옵니다.
검색어를 바꾸면 다른 주제도 모을 수 있어요. (실습이니 논문 수는 적게!)

In [ ]:
import requests, xmltodict, time, pandas as pd

PUBMED_EMAIL = 'student@example.com'   # NCBI 권장: 아무 이메일
QUERIES = [
    '"monoclonal antibody" AND "aggregation" AND "formulation"',
    '"monoclonal antibody" AND "viscosity"',
]
MAX_PER_QUERY = 40   # 쿼리당 최대 논문 수 (실습용으로 작게)

def search_pubmed(query, retmax=40):
    url = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi'
    p = {'db':'pubmed','term':query,'retmax':retmax,'retmode':'json','email':PUBMED_EMAIL}
    r = requests.get(url, params=p, timeout=30); r.raise_for_status()
    return r.json().get('esearchresult',{}).get('idlist',[])

def fetch_abstracts(pmids):
    url = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi'
    out = []
    for i in range(0, len(pmids), 100):
        batch = pmids[i:i+100]
        p = {'db':'pubmed','id':','.join(batch),'rettype':'xml','retmode':'xml','email':PUBMED_EMAIL}
        r = requests.get(url, params=p, timeout=60); r.raise_for_status()
        arts = xmltodict.parse(r.text).get('PubmedArticleSet',{}).get('PubmedArticle',[])
        if isinstance(arts, dict): arts = [arts]
        for a in arts:
            try:
                mc = a['MedlineCitation']
                pmid = mc['PMID']['#text'] if isinstance(mc['PMID'], dict) else mc['PMID']
                art = mc['Article']
                title = art.get('ArticleTitle','')
                if isinstance(title, dict): title = title.get('#text','')
                ab = art.get('Abstract',{}).get('AbstractText','')
                if isinstance(ab, list):
                    ab = ' '.join([x.get('#text','') if isinstance(x, dict) else str(x) for x in ab])
                elif isinstance(ab, dict): ab = ab.get('#text','')
                year = art.get('Journal',{}).get('JournalIssue',{}).get('PubDate',{}).get('Year','N/A')
                if ab and len(str(ab)) > 50:
                    out.append({'pmid':pmid,'title':str(title),'abstract':str(ab),'year':year})
            except Exception:
                continue
        time.sleep(0.4)   # NCBI 예의: 너무 빨리 요청하지 않기
    return out

pmids = set()
for q in QUERIES:
    found = search_pubmed(q, MAX_PER_QUERY)
    pmids.update(found)
    print(f'  검색 {len(found):>3}건 | {q}')
    time.sleep(0.4)

articles = fetch_abstracts(list(pmids))
df1 = pd.DataFrame(articles)
df1.to_csv('step1_abstracts.csv', index=False, encoding='utf-8-sig')
print(f'\n✅ 수집 완료: {len(df1)}편  → step1_abstracts.csv 저장')
df1.head(3)

## STEP 2 — GPT로 '안정성 관련 논문'만 골라내기 (필터링)

검색만으로는 엉뚱한 논문도 섞입니다(예: 항체를 단순 '검출 시약'으로만 쓴 논문).
GPT에게 제목+초록을 보여주고 **관련=1 / 비관련=0** 을 판단하게 합니다.

> 🔑 아래 셀에서 OpenAI 키를 넣습니다. **키가 없으면 그냥 Enter** → 필터를 건너뛰고 전체를 다음 단계로 넘깁니다.

In [ ]:
import os
from getpass import getpass

OPENAI_KEY = ''
try:
    from google.colab import userdata        # Colab의 비밀키 저장소
    OPENAI_KEY = userdata.get('OPENAI_API_KEY') or ''
except Exception:
    OPENAI_KEY = os.environ.get('OPENAI_API_KEY','')
if not OPENAI_KEY:
    OPENAI_KEY = getpass('OpenAI API key 입력 (없으면 그냥 Enter): ').strip()

USE_GPT = bool(OPENAI_KEY)
MODEL   = 'gpt-4o-mini'   # 저렴하고 충분히 똑똑한 모델 (연구 원본은 gpt-5-mini 사용)
print('🟢 GPT 사용 모드' if USE_GPT else '⚪ 키 없음 → STEP 2·3 건너뜀, STEP 4에서 샘플 데이터 사용')

In [ ]:
if USE_GPT:
    from openai import OpenAI
    import json, time
    client = OpenAI(api_key=OPENAI_KEY)

    FILTER_SYS = (
        'You are an expert in monoclonal antibody (mAb) stability. '
        'Decide if the article contains mechanistic information about mAb stability '
        '(aggregation, viscosity, oxidation, deamidation, formulation effects, stress effects, '
        'immunogenicity from instability, etc.). '
        'Answer ONLY a JSON object: {"relevant": true/false, "reason": "one short sentence"}.'
    )

    def is_relevant(title, abstract):
        try:
            r = client.chat.completions.create(
                model=MODEL,
                messages=[{'role':'system','content':FILTER_SYS},
                          {'role':'user','content':f'Title: {title}\nAbstract: {abstract[:1500]}'}],
                max_completion_tokens=120)
            txt = r.choices[0].message.content.replace('```json','').replace('```','').strip()
            return bool(json.loads(txt).get('relevant', True))
        except Exception as e:
            print('  (판단 실패, 일단 통과)', str(e)[:60]); return True

    flags = []
    for i, row in df1.iterrows():
        flags.append(1 if is_relevant(row['title'], row['abstract']) else 0)
        time.sleep(0.2)
    df1['relevant'] = flags
    df2 = df1[df1['relevant'] == 1].copy()
    print(f'✅ 관련 논문 {len(df2)} / 전체 {len(df1)}')
else:
    df2 = df1.copy()
    print('⚪ 필터 건너뜀 → 전체', len(df2), '편을 그대로 사용')

## STEP 3 — GPT로 인과관계 (원인 → 관계 → 결과) 추출

이 단계가 파이프라인의 **핵심**입니다. 초록 한 편에서 다음 같은 삼중항(triplet)을 뽑아냅니다.

| cause(원인) | relationship(관계) | effect(결과) |
|---|---|---|
| thermal stress | promotes | aggregation |
| polysorbate 80 | inhibits | aggregation |
| low pH | promotes | deamidation |

각 노드는 6개 카테고리(sequence/structure/formulation/stress/stability/quality_outcome) 중 하나로 분류됩니다.

> 💰 비용을 아끼려고 **앞쪽 20편만** 처리합니다. (키가 없으면 이 셀은 자동으로 건너뜁니다.)

In [ ]:
LIMIT = 20   # 비용 절약: 관련 논문 중 앞 20편만

EXTRACT_SYS = (
    'You are a pharmaceutical scientist. Extract mechanistic relationships about mAb stability '
    'from the abstract as directed triplets.\n'
    '6 node categories: sequence, structure, formulation, stress, stability, quality_outcome.\n'
    'Relation types: stabilizes, destabilizes, increases, decreases, inhibits, promotes, induces, '
    'prevents, requires, modifies, binds, shields, aggregates, oxidizes, deamidates, isomerizes, '
    'fragments, unfolds, adsorbs, precipitates, degrades, correlates.\n'
    'Return ONLY JSON: {"relations":[{"cause":"","category_cause":"","effect":"",'
    '"category_effect":"","relationship":"","confidence":"high/medium/low"}]}.'
)

def extract_relations(title, abstract):
    try:
        r = client.chat.completions.create(
            model=MODEL,
            messages=[{'role':'system','content':EXTRACT_SYS},
                      {'role':'user','content':f'Title: {title}\nAbstract: {abstract[:2500]}'}],
            max_completion_tokens=1500)
        txt = r.choices[0].message.content.replace('```json','').replace('```','').strip()
        return json.loads(txt).get('relations', [])
    except Exception as e:
        print('  (추출 실패)', str(e)[:60]); return []

rows = []
if USE_GPT:
    for i, row in df2.head(LIMIT).iterrows():
        for rel in extract_relations(row['title'], row['abstract']):
            rel['pmid'] = row['pmid']
            rows.append(rel)
        time.sleep(0.2)
    raw = pd.DataFrame(rows)
    print(f'✅ 추출된 관계: {len(raw)}개')
    display(raw.head(8))
else:
    print('⚪ 키가 없어 추출을 건너뜁니다. STEP 4에서 공개 샘플 데이터로 그래프를 그립니다.')

### STEP 3-b — 표준화(간단 버전) + 엣지 집계

같은 뜻의 표현을 하나로 모으고(여기선 소문자/공백 정리만), 같은 (원인,관계,결과)가 몇 번 나왔는지 세어
**엣지 요약표**를 만듭니다. 이 표가 그래프의 재료이자, 실제 연구에서는 GATED 모델의 학습 데이터가 됩니다.

> 🔎 원본 연구에서는 이 표준화도 GPT가 수행해 수만 개의 표현을 수백 개 표준 용어로 정리합니다(STEP 3 노트북 참고).

In [ ]:
import os
if USE_GPT and len(rows) > 0:
    raw['cause']  = raw['cause'].astype(str).str.lower().str.strip()
    raw['effect'] = raw['effect'].astype(str).str.lower().str.strip()
    edges = (raw.groupby(['cause','category_cause','effect','category_effect','relationship'])
                .size().reset_index(name='frequency')
                .sort_values('frequency', ascending=False))
    edges['num_papers'] = edges['frequency']   # 실습 단순화
    edges.to_csv('step3_edges.csv', index=False, encoding='utf-8-sig')
    print(f'✅ 내 엣지표: {len(edges)}개 → step3_edges.csv 저장')
    display(edges.head(10))
else:
    print('⚪ 추출 결과가 없어 엣지표를 만들지 않았습니다(샘플로 진행).')

## STEP 4 — 네트워크 그래프 그리기 🎨

이제 엣지표를 **그래프**로 그립니다. 먼저 그릴 데이터를 정합니다.
- 내가 GPT로 만든 `step3_edges.csv` 가 있으면 그것을 사용
- 없으면 연구팀이 공개한 **실제 전체 그래프 데이터(샘플)** 를 내려받아 사용

In [ ]:
import os, pandas as pd
SAMPLE_URL = 'https://raw.githubusercontent.com/STARG-LEE/mab-causal-network-v2/main/step2c_v4d_causal_edges_summary.csv'

if os.path.exists('step3_edges.csv'):
    g = pd.read_csv('step3_edges.csv')
    SRC = '내가 GPT로 만든 데이터'
else:
    g = pd.read_csv(SAMPLE_URL)
    SRC = '연구팀 공개 샘플(전체 그래프)'
print(f'그래프 데이터: {SRC} — 엣지 {len(g):,}개')
g.head(5)

### 색 규칙 (웹 시각화 index.html 과 동일)
- **노드 색 = 6개 카테고리** · **노드 크기 = 등장 빈도**
- **화살표 색 = 방향성**: 🟢 초록=안정화/억제, 🔴 빨강=불안정/촉진, ⚪ 회색=중립

In [ ]:
%matplotlib inline
import networkx as nx, matplotlib.pyplot as plt
from matplotlib.lines import Line2D
try:
    import koreanize_matplotlib            # 한글 깨짐 방지(자동으로 한글 폰트 적용)
except Exception:
    for _f in ['Malgun Gothic','AppleGothic','NanumGothic']:   # 폴백
        try: plt.rcParams['font.family']=_f; break
        except Exception: pass
plt.rcParams['axes.unicode_minus']=False

CAT_COLOR = {'sequence':'#c4b5fd','structure':'#a78bfa','formulation':'#34d399',
             'stress':'#fbbf24','stability':'#f87171','quality_outcome':'#fb923c'}
POSITIVE = {'stabilizes','decreases','inhibits','prevents','shields'}
NEGATIVE = {'destabilizes','increases','promotes','induces','aggregates','oxidizes',
            'deamidates','isomerizes','fragments','unfolds','adsorbs','precipitates','degrades','denatures'}
def rel_color(r):
    if r in POSITIVE: return '#22c55e'
    if r in NEGATIVE: return '#ef4444'
    return '#94a3b8'

# 노드 카테고리 / 빈도 사전
node_cat, strength = {}, {}
for _, r in g.iterrows():
    node_cat.setdefault(str(r['cause']),  str(r['category_cause']))
    node_cat.setdefault(str(r['effect']), str(r['category_effect']))
    strength[str(r['cause'])]  = strength.get(str(r['cause']),0)  + int(r['frequency'])
    strength[str(r['effect'])] = strength.get(str(r['effect']),0) + int(r['frequency'])

# 너무 빽빽하면 가독성↓ → 빈도 상위 100개 엣지만
sub = g.sort_values('frequency', ascending=False).head(100)
G = nx.DiGraph()
for _, r in sub.iterrows():
    G.add_edge(str(r['cause']), str(r['effect']), rel=str(r['relationship']), w=int(r['frequency']))

pos = nx.spring_layout(G, k=0.9, iterations=150, seed=42)
fig, ax = plt.subplots(figsize=(14,10)); fig.patch.set_facecolor('#0a0e17'); ax.set_facecolor('#0a0e17')
nx.draw_networkx_edges(G,pos,edge_color=[rel_color(G[u][v]['rel']) for u,v in G.edges()],
                       width=[0.5+G[u][v]['w']*0.1 for u,v in G.edges()],alpha=0.55,
                       arrows=True,arrowsize=9,connectionstyle='arc3,rad=0.08',ax=ax)
nx.draw_networkx_nodes(G,pos,node_color=[CAT_COLOR.get(node_cat.get(n,''),'#888') for n in G.nodes()],
                       node_size=[120+strength.get(n,0)*3 for n in G.nodes()],
                       edgecolors='#0a0e17',linewidths=0.5,ax=ax)
big = sorted(G.nodes(), key=lambda n:-strength.get(n,0))[:20]
nx.draw_networkx_labels(G,pos,labels={n:n for n in big},font_size=8,font_color='#e8ecf4',ax=ax)
ax.set_title('내 mAb 안정성 인과 네트워크 (빈도 상위 100 엣지)',color='#e8ecf4',fontsize=14,fontweight='bold')
ax.legend(handles=[Line2D([0],[0],marker='o',color='w',markerfacecolor=c,markersize=10,label=k) for k,c in CAT_COLOR.items()],
          loc='upper left',framealpha=0.2,labelcolor='#e8ecf4',fontsize=8)
ax.axis('off'); plt.tight_layout(); plt.savefig('my_network.png',dpi=150,facecolor='#0a0e17'); plt.show()
print('🖼️  my_network.png 저장 완료')

### (선택) 살아있는 웹 시각화에 내 CSV 올려보기

연구팀의 **인터랙티브 웹 그래프**에는 *CSV 업로드* 기능이 있습니다.
방금 만든 `step3_edges.csv`(또는 `my_network` 데이터)를 그대로 올리면 마우스로 끌고 확대/필터링할 수 있어요.

🔗 **https://starg-lee.github.io/mab-causal-network-v2/**  → 왼쪽 패널의 *Upload CSV* 클릭 → 내 CSV 선택

필요한 컬럼: `cause, category_cause, effect, category_effect, relationship, frequency, num_papers`

## STEP 5 (개념만) — GATED 모델은 무엇을 할까?

여기까지가 **직접 실습**입니다. 실제 연구는 이 엣지표로 **mAb-GATED** 라는 AI 모델을 학습시킵니다.

**한 줄 요약:** 그래프에서 정답 노드 하나를 가린 뒤("빈칸 채우기"), 주변 이웃 노드들을 보고
가려진 **안정성 지표**가 무엇인지 맞히도록 학습합니다.

```
입력 : (formulation, stress, structure ... 6개 카테고리의 이웃 노드들)
   ↓  PubMedBERT 임베딩으로 단어 의미 초기화
   ↓  GAT (그래프 이웃 정보 모으기)
   ↓  Transformer 인코더-디코더
정답 : 가려진 stability 노드  (예: aggregation? viscosity? oxidation?)
```

성능(원본 논문, 테스트셋): **MRR 0.88, Hits@1 84.6%** — 빈도/단순기법 기반 베이스라인을 크게 능가.
자세한 구조와 결과는 **교안**과 `step4_mab_gated_colab.ipynb` 원본을 참고하세요.

---
### 🏁 도전 과제
1. STEP 1의 검색어를 바꿔 다른 주제(예: `oxidation`, `freeze-thaw`)로 나만의 그래프를 만들어 보세요.
2. STEP 4에서 `aggregation` 한 노드의 이웃만 골라 그려 보세요(`02_그래프_그리기.ipynb` 참고).
3. 초록=좋은 방향, 빨강=나쁜 방향. 내 그래프에서 빨강 화살표가 가장 많이 향하는 노드는 무엇인가요?